In [1]:
!pip install mediapipe

In [2]:
pip install pyserial

Note: you may need to restart the kernel to use updated packages.


In [1]:
import serial.tools.list_ports

# List all available serial ports
ports = serial.tools.list_ports.comports()

for port, desc, hwid in sorted(ports):
    print(f"Port: {port}, Description: {desc}, Hardware ID: {hwid}")

Port: /dev/ttyS0, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS1, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS2, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS3, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS4, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS5, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS6, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS7, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS8, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS9, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS10, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS11, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS12, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS13, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS14, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS15, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS16, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS17, Description: n/a, Hardware ID: n/a
Port: /dev/ttyS18, Description: n/a, H

In [2]:
import serial 
import time 

ser = serial.Serial('/dev/ttyUSB0',baudrate=115200,bytesize =8, parity ='N', stopbits =1)
hex = '3A0100020003000400'

data_send = bytes.fromhex(hex)
ser.write(data_send)

ser.close()

USE THIS FOR DEMO

In [ ]:
import cv2
import mediapipe as mp
import threading
import serial
import time

# Initialize serial communication
ser = serial.Serial('/dev/ttyUSB0', 115200)

# Initialize webcam
webcam = cv2.VideoCapture(2)

# Initialize Mediapipe hands module
mpHands = mp.solutions.hands
hands = mpHands.Hands()
mpDraw = mp.solutions.drawing_utils

# Global variables for command and cooldown management
last_activation_time = 0
cooldown_duration = 1  # Cooldown duration in seconds

# Function to send command via serial with timed activation
def send_command(command, duration):
    global last_activation_time

    # Calculate current time
    current_time = time.time()

    # Check cooldown
    if current_time - last_activation_time >= cooldown_duration:
        # Send command
        ser.write(bytes.fromhex(command))
        print(f"Sent command {command}")
        last_activation_time = current_time

# Determine if the hand is facing the camera (palm) or away (back of hand)
def is_palm_facing(lm_list):
    wrist_y = lm_list[0][2]  # y-coordinate of wrist
    base_finger_y = lm_list[9][2]  # y-coordinate of the middle finger base
    return wrist_y < base_finger_y  # Palm facing if wrist is below the base of the fingers

def finger_counting(imageFrame):
    # Convert frame to RGB (Mediapipe requires RGB input)
    imgRGB = cv2.cvtColor(imageFrame, cv2.COLOR_BGR2RGB)

    # Process the frame with Mediapipe hands
    results = hands.process(imgRGB)

    # Initialize finger count
    finger_count = 0

    # If hands are detected
    if results.multi_hand_landmarks:
        for hand_landmark, hand_info in zip(results.multi_hand_landmarks, results.multi_handedness):
            # Get hand label (Left or Right)
            hand_label = hand_info.classification[0].label
            fingers = [False] * 5  # Initialize a list to track each finger
            lm_list = []

            # Extract landmark positions
            for id, lm in enumerate(hand_landmark.landmark):
                h, w, _ = imageFrame.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                lm_list.append([id, cx, cy])

            # Determine hand orientation (palm facing or back of hand facing)
            palm_facing = is_palm_facing(lm_list)

            if palm_facing:
                # Palm facing: regular finger checking logic
                if hand_label == "Right":
                    # For the right hand, thumb tip should be on the right side of the IP joint
                    if lm_list[4][1] > lm_list[3][1]:  # Thumb
                        fingers[0] = True
                else:  # Left hand
                    # For the left hand, thumb tip should be on the left side of the IP joint
                    if lm_list[4][1] < lm_list[3][1]:  # Thumb
                        fingers[0] = True
            else:
                # Back of hand facing: reversed logic for thumb
                if hand_label == "Right":
                    # For the right hand, thumb tip should be on the left side of the IP joint
                    if lm_list[4][1] < lm_list[3][1]:  # Thumb
                        fingers[0] = True
                else:  # Left hand
                    # For the left hand, thumb tip should be on the right side of the IP joint
                    if lm_list[4][1] > lm_list[3][1]:  # Thumb
                        fingers[0] = True

            # Common logic for index to little finger (same for both hands and orientations)
            if lm_list[8][2] < lm_list[7][2]:  # Index finger
                fingers[1] = True
            if lm_list[12][2] < lm_list[11][2]:  # Middle finger
                fingers[2] = True
            if lm_list[16][2] < lm_list[15][2]:  # Ring finger
                fingers[3] = True
            if lm_list[20][2] < lm_list[19][2]:  # Little finger
                fingers[4] = True

            # Count how many fingers are extended
            finger_count = sum(fingers)

            # Draw circles at finger tips
            for lm in lm_list:
                cv2.circle(imageFrame, (lm[1], lm[2]), 7, (255, 0, 0), cv2.FILLED)

            # Draw hand landmarks and connections
            mpDraw.draw_landmarks(imageFrame, hand_landmark, mpHands.HAND_CONNECTIONS)

    # Display finger count on the frame
    cv2.putText(imageFrame, f"Finger Count: {finger_count}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 98), 3)

    # Send commands based on finger count
    if finger_count == 5:
        threading.Thread(target=send_command, args=('3A0100020003000400', 5)).start()  
    elif finger_count == 4:
        threading.Thread(target=send_command, args=('3A0100020003000401', 4)).start()  # Turn on LAMP 4
    elif finger_count == 3:
        threading.Thread(target=send_command, args=('3A0100020003010400', 3)).start()  # Turn on LAMP 3
    elif finger_count == 2:
        threading.Thread(target=send_command, args=('3A0100020103000400', 2)).start()  # Turn on LAMP 2
    elif finger_count == 1:
        threading.Thread(target=send_command, args=('3A0101020003000400', 1)).start()  # Turn on LAMP 1
    else:
        threading.Thread(target=send_command, args=('3A0100020003000400', 0)).start()  # Turn off immediately

# Main loop
while True:
    _, imageFrame = webcam.read()

    # Perform finger counting and command sending
    finger_counting(imageFrame)

    # Display the resulting frame
    cv2.imshow("Hand Gesture Finger Count", imageFrame)

    # Exit if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
webcam.release()
cv2.destroyAllWindows()

2024-09-29 08:21:58.782305: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-09-29 08:21:58.786996: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-29 08:21:58.829421: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-29 08:21:58.876086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-29 08:21:58.915381: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

Sent command 3A0100020003000400


Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400


W0000 00:00:1727569330.982948    4619 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
/home/gott/openvino_env/lib/python3.12/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Sent command 3A0100020003000400
Sent command 3A0101020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent command 3A0100020003000400
Sent com